# Data Cleaning: Messy Customer Dataset
**Objective:** Take a deliberately messy dataset and turn it into a clean, analysis-ready one, documenting every decision.

**Tech stack:** Python, pandas, numpy

> **Note:** `messy_customer_data.csv` is a practice dataset with planted problems (missing values, duplicates, inconsistent text, mixed date formats, money stored as text, impossible values). For your final submission, repeat the same steps on a real messy Kaggle dataset (search "data cleaning").

## 0. Setup

In [8]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

## 1. Load data and data quality report
_Checklist: nulls per column, duplicate rows, data type issues, value range anomalies_

In [9]:
raw = pd.read_csv("messy_customer_data.csv")
df = raw.copy()   # work on a copy so the raw data stays untouched
df.head(10)

,Customer_ID,Name,Gender,Age,City,Join_Date,Annual_Income,Purchase_Amount,Satisfaction_Score,Email
0,11000,Vikram Reddy,Female,55.0,BENGALURU,17-Mar-2024,"$34,400",587.98,2.0,vikram.reddy@example.com
1,10195,Priya Iyer,female,44.0,Hyderabad,06-Mar-2023,NaN,419.42,5.0,priya.iyer@example.com
2,10753,Ananya Singh,male,55.0,CHENNAI,19-Jul-2024,"$80,100",433.43,7.0,ananya.singh@example.com
3,10440,Lakshmi Sharma,NaN,42.0,Delhi,19-Aug-2021,"$84,500",2209.03,8.0,lakshmi.sharma@example.com
4,10622,Ananya Reddy,male,35.0,Chennai,28-Aug-2023,"$51,100",921.24,4.0,ananya.reddy@example.com
5,10557,Pooja Gupta,m,29.0,PUNE,2021-08-28,"$49,400",1091.65,8.0,pooja.gupta@example.com
6,10156,Pooja Gupta,F,38.0,Mumbai,2021-01-13,"$55,400",1376.10,4.0,pooja.gupta@example.com
7,10433,Nikhil Singh,Female,42.0,Hyderabad,2022-10-18,"$18,600",343.67,NaN,nikhil.singh@example.com
8,10691,Nikhil Nair,MALE,28.0,kolkata,2022-09-07,"$66,900",547.33,9.0,nikhil.nair@example.com
9,10233,Karthik Menon,Female,28.0,CHENNAI,2022-04-27,"$102,200",2253.04,4.0,NaN


In [10]:
print("Shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
print()
print("Nulls per column:")
print(df.isnull().sum())
print()
print("Data types:")
print(df.dtypes)

Shape: (1060, 10)
Duplicate rows: 60

Nulls per column:
Customer_ID             0
Name                    0
Gender                 32
Age                    75
City                   42
Join_Date               0
Annual_Income          84
Purchase_Amount         0
Satisfaction_Score    127
Email                  50
dtype: int64

Data types:
Customer_ID             int64
Name                      str
Gender                    str
Age                   float64
City                      str
Join_Date                 str
Annual_Income             str
Purchase_Amount       float64
Satisfaction_Score    float64
Email                     str
dtype: object


In [11]:
# Spot inconsistent categories
print(df["Gender"].value_counts(dropna=False))
print()
print(df["City"].value_counts(dropna=False))

Gender
male      128
female    116
FEMALE    116
Male      103
f         102
m          98
F          97
MALE       92
M          89
Female     87
NaN        32
Name: count, dtype: int64

City
Delhi          47
Mumbai         45
KOLKATA        45
PUNE           44
delhi          44
CHENNAI        43
 Pune          42
NaN            42
HYDERABAD      42
BENGALURU      41
 Mumbai        41
MUMBAI         40
 Delhi         38
pune           38
 Hyderabad     36
Chennai        35
 Bengaluru     34
kolkata        33
Pune           33
bengaluru      32
hyderabad      32
 Chennai       31
Bengaluru      31
 Kolkata       30
chennai        30
mumbai         30
Hyderabad      29
DELHI          28
Kolkata        24
Name: count, dtype: int64


In [12]:
# Value range anomalies in numeric columns
df[["Age", "Purchase_Amount", "Satisfaction_Score"]].describe()

,Age,Purchase_Amount,Satisfaction_Score
count,985.000000,1060.000000,933.000000
mean,37.290355,1146.089613,5.516613
std,45.046599,2697.969230,2.899051
min,-5.000000,82.370000,1.000000
25%,28.000000,531.395000,3.000000
50%,35.000000,855.735000,6.000000
75%,42.000000,1315.780000,8.000000
max,999.000000,50814.400000,10.000000


**Data quality issues found:** _List every problem you can see, for example:_
- _Gender has many spellings of the same value_
- _Age contains impossible values_
- _..._

In [13]:
# Save a 'before' snapshot for the final comparison table
before = {
    "Rows": len(raw),
    "Total nulls": int(raw.isnull().sum().sum()),
    "Duplicate rows": int(raw.duplicated().sum()),
}

## 2. Remove duplicates
_Checklist: identify and remove duplicate rows; document how many were removed_

In [14]:
n_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {n_before - len(df)} duplicate rows. Rows now: {len(df)}")

Removed 60 duplicate rows. Rows now: 1000


**Note:** _Why is it safe to drop these? Think about whether two different customers could have identical rows._

## 3. Standardise formats
_Checklist: normalise inconsistent text; date formats to datetime_

In [15]:
# Gender: map every variant to Male / Female
gender_map = {"male": "Male", "m": "Male", "female": "Female", "f": "Female"}
df["Gender"] = df["Gender"].str.strip().str.lower().map(gender_map)
print(df["Gender"].value_counts(dropna=False))

Gender
Female    486
Male      484
NaN        30
Name: count, dtype: int64


In [16]:
# City: strip whitespace and use consistent capitalisation
df["City"] = df["City"].str.strip().str.title()
print(df["City"].value_counts(dropna=False))

City
Pune         151
Mumbai       146
Delhi        145
Chennai      133
Hyderabad    130
Bengaluru    128
Kolkata      127
NaN           40
Name: count, dtype: int64


In [17]:
# Join_Date: several text formats -> datetime
df["Join_Date"] = pd.to_datetime(df["Join_Date"], format="mixed")
print(df["Join_Date"].dtype)
df["Join_Date"].head()

datetime64[us]


0   2024-03-17
1   2023-03-06
2   2024-07-19
3   2021-08-19
4   2023-08-28
Name: Join_Date, dtype: datetime64[us]

## 4. Fix data types
_Checklist: dates as datetime, IDs as string, monetary values as float_

In [18]:
# Customer_ID is an identifier, not a number to do maths on
df["Customer_ID"] = df["Customer_ID"].astype(str)

# Annual_Income is text like "$45,000" -> float
df["Annual_Income"] = (
    df["Annual_Income"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)
df.dtypes

Customer_ID                      str
Name                             str
Gender                           str
Age                          float64
City                             str
Join_Date             datetime64[us]
Annual_Income                float64
Purchase_Amount              float64
Satisfaction_Score           float64
Email                            str
dtype: object

## 5. Outlier detection
_Checklist: IQR or Z-score; decide whether to cap, remove, or retain each column_

In [19]:
def iqr_bounds(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

for col in ["Age", "Annual_Income", "Purchase_Amount"]:
    lo, hi = iqr_bounds(df[col].dropna())
    n_out = ((df[col] < lo) | (df[col] > hi)).sum()
    print(f"{col}: bounds=({lo:.0f}, {hi:.0f}), outliers={n_out}")

Age: bounds=(7, 63), outliers=7
Annual_Income: bounds=(7675, 103475), outliers=9
Purchase_Amount: bounds=(-642, 2482), outliers=13


**Decision per column:** _For each column decide: remove, cap, or keep, and say why._
- **Age:** ...
- **Annual_Income:** ...
- **Purchase_Amount:** ...

In [20]:
# Age: values outside 18-100 are impossible -> set to NaN, then they get imputed in Section 6
df.loc[(df["Age"] < 18) | (df["Age"] > 100), "Age"] = np.nan

# Income and purchase amount: cap extreme values at the IQR upper bound (winsorise)
for col in ["Annual_Income", "Purchase_Amount"]:
    lo, hi = iqr_bounds(df[col].dropna())
    df[col] = df[col].clip(lower=lo, upper=hi)

## 6. Handle missing values
_Checklist: choose a strategy per column and justify each in a markdown cell_

In [21]:
df.isnull().sum()

Customer_ID             0
Name                    0
Gender                 30
Age                    76
City                   40
Join_Date               0
Annual_Income          80
Purchase_Amount         0
Satisfaction_Score    120
Email                  50
dtype: int64

In [22]:
# Numeric columns -> median (robust to outliers)
for col in ["Age", "Annual_Income", "Satisfaction_Score"]:
    df[col] = df[col].fillna(df[col].median())

# Categorical columns -> mode
for col in ["Gender", "City"]:
    df[col] = df[col].fillna(df[col].mode()[0])

# Email cannot be guessed -> fill with an explicit marker instead of inventing data
df["Email"] = df["Email"].fillna("unknown")

df.isnull().sum()

Customer_ID           0
Name                  0
Gender                0
Age                   0
City                  0
Join_Date             0
Annual_Income         0
Purchase_Amount       0
Satisfaction_Score    0
Email                 0
dtype: int64

**Justification per column:** _Replace the lines below with your own reasoning._
- **Age / Annual_Income / Satisfaction_Score (median):** ...
- **Gender / City (mode):** ...
- **Email (marker):** ...

_Think about the alternative: when would deleting rows be better than imputing?_

## 7. Before vs. after summary
_Checklist: null count, duplicate count, row count, dtype accuracy_

In [23]:
after = {
    "Rows": len(df),
    "Total nulls": int(df.isnull().sum().sum()),
    "Duplicate rows": int(df.duplicated().sum()),
}
summary = pd.DataFrame({"Before": before, "After": after})
summary

,Before,After
Rows,1060,1000
Total nulls,410,0
Duplicate rows,60,0


In [24]:
print("Dtypes before:\n", raw.dtypes, "\n")
print("Dtypes after:\n", df.dtypes)

Dtypes before:
 Customer_ID             int64
Name                      str
Gender                    str
Age                   float64
City                      str
Join_Date                 str
Annual_Income             str
Purchase_Amount       float64
Satisfaction_Score    float64
Email                     str
dtype: object 

Dtypes after:
 Customer_ID                      str
Name                             str
Gender                           str
Age                          float64
City                             str
Join_Date             datetime64[us]
Annual_Income                float64
Purchase_Amount              float64
Satisfaction_Score           float64
Email                            str
dtype: object


## 8. Save the cleaned dataset

In [25]:
df.to_csv("cleaned_customer_data.csv", index=False)
print("Saved cleaned_customer_data.csv with shape", df.shape)

Saved cleaned_customer_data.csv with shape (1000, 10)


## 9. Conclusion
_Write 4-6 sentences: what were the biggest problems, what choices did you make, and how could the cleaned data now be used?_